In [3]:
import torch
import torch.nn as nn
import einops
import tiktoken
from torch.nn import functional as F
from dataclasses import dataclass

device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
print(device)

tokenizer = tiktoken.get_encoding('gpt2')

cuda


main changes:
* dataloader class
* wte = proj
* initialization stddev
* residual $\frac{1}{\sqrt{N}}$

In [4]:
@dataclass
class GPTConfig:
    batch_size: int = 32
    block_size: int = 256
    vocab_size: int = tokenizer.n_vocab # 50257, 50000 merges + 256 byte + <endoftext>
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384
    lr: float = 3e-4

In [5]:
class DataLoader():
    def __init__(self, B, T):
        self.B = B
        self.T = T

        with open('input.txt', 'r', encoding='utf-8') as f:
            text = f.read()

        data = torch.tensor(tokenizer.encode(text), dtype=torch.long)
        n = int(0.9*len(data)) # first 90% will be train, rest val
        self.train_data = data[:n]
        self.val_data = data[n:]

        self.current_pos = 0
        print(f"tokens: {len(data)}")
        print(f"1 epoch is {len(self.train_data) // B * T} iters")

    def get_batch(self, split):
        data = self.train_data if split == 'train' else self.val_data
        ix = torch.randint(low=0, high=len(data)-GPTConfig().block_size, size=(GPTConfig().batch_size, ))

        x = torch.stack([data[i:i+GPTConfig().block_size] for i in ix])
        y = torch.stack([data[i+1:i+GPTConfig().block_size+1] for i in ix])
        x, y = x.to(device), y.to(device)
        return x, y
    
    def next_batch(self, split):
        B, T, = self.B, self.T
        data = self.train_data if split == 'train' else self.val_data
        buffer = data[self.current_pos : self.current_pos + B*T + 1]

        x = buffer[:-1].view(B, T)
        y = buffer[1:].view(B, T)
        x, y = x.to(device), y.to(device)

        self.current_pos += B*T
        return x, y



In [13]:
loader = DataLoader(GPTConfig.batch_size, GPTConfig.block_size) # GPTConfig.block_size
loader.next_batch('train')

tokens: 338025
1 epoch is 2433536 iters


(tensor([[ 5962, 22307,    25,  ...,   198,   454,   279],
         [ 7938,    11,   304,  ...,    11,   351, 18201],
         [   11,   284, 15867,  ...,  1337,   198, 11980],
         ...,
         [  198,  2504,  1111,  ...,   198, 40569, 25690],
         [ 2937,    25,   198,  ...,   318,   339,    30],
         [  869,   683, 37706,  ...,  3886,   262,  2910]], device='cuda:0'),
 tensor([[22307,    25,   198,  ...,   454,   279,  7938],
         [   11,   304,   260,  ...,   351, 18201,    11],
         [  284, 15867,   287,  ...,   198, 11980,   262],
         ...,
         [ 2504,  1111,   674,  ..., 40569, 25690,  2937],
         [   25,   198,   198,  ...,   339,    30,   869],
         [  683, 37706,    13,  ...,   262,  2910,   356]], device='cuda:0'))

In [7]:
class FFN(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(approximate='tanh'),
            nn.Linear(4 * config.n_embd, config.n_embd)
        )
    
    def forward(self, x):
        return x + self.net(x)

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.ln1 = nn.LayerNorm(config.n_embd)
        self.multiattn = nn.MultiheadAttention(embed_dim=config.n_embd, num_heads=config.n_head, batch_first=True)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.ffn = FFN(config)
    
    def forward(self, x):
        norm = self.ln1(x)
        attn, _ = self.multiattn(query=norm, key=norm, value=norm, need_weights=False, attn_mask=torch.triu(torch.ones(x.shape[-2], x.shape[-2], device=device, dtype=torch.bool), diagonal=1))
        x = x + attn
        x = x + self.ffn(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln = nn.LayerNorm(config.n_embd)
        ))

        self.proj = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.proj.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.config.block_size, f"context length of {T} exceeds block_size of {self.config.block_size}"
        
        pos_emb = self.transformer.wpe(torch.arange(T, dtype=torch.long, device=device)) # (block, embed)
        tok_emb = self.transformer.wte(idx) # (B,T,C) -> (batch, block, embed)
        x = tok_emb + pos_emb

        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln(x)

        if targets is None:
            loss = None
            logits = x[:, [-1], :] # only need to calc last ones for generation
            logits = self.proj(logits)
        else:
            logits = self.proj(x)
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)    
    
        return logits, loss


    def generate(self, idx, max_tokens=1, temp=1):
        for _ in range(max_tokens):
            logits, _ = self(idx[:,-self.config.block_size:])
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=1)
            ix = torch.multinomial(probs / temp, num_samples=1)
            idx = torch.cat((idx, ix), dim=1) # lol no need to cut
            # print(decode(ix[0].tolist()), end="", flush=True)
        return idx

In [8]:
model = GPT(GPTConfig())
model = model.to(device=device)

optimizer = torch.optim.AdamW(model.parameters(), lr=model.config.lr)

In [14]:
for i in range(100): 
    Xb, Yb = loader.next_batch('train')

    with torch.autocast("cuda", dtype=torch.bfloat16):
        logits, loss = model(Xb, Yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
    if i % 10 == 0:
        print(f"step: {i}, loss: {loss.item():4f}")

step: 0, loss: 7.103582


KeyboardInterrupt: 